# Coursework 1 - Overall ML Pipeline (Stroke Prediction)

**Dataset:** healthcare-dataset-stroke-data.csv (Kaggle / UCU format)

This notebook presents a structure machine learning workflow for stroke predicition using logistic regression.
The focus is on rigorous experiment design, including dataset exploration, preprocessing, model training and validation, and performance evaluation.
Attention is given to reproducibility, imterpretability and computational efficiency with visualisations supporting the analysis and clear justification of modelling choices.


# Set up and Imports

In [ ]:
# Python core and scientific libraries
import time 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

# Model selection and evaluation tools
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, learning_curve

# Pipeline and preprocessing
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# Models
from sklearn.linear_model import Ridge, LogisticRegression

# Metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# Reproducibility
RNG = 42
np.random.seed(RNG)

# Load Dataset

In [11]:
DATA_PATH = r"healthcare-dataset-stroke-data.csv" # Set the path to the stroke dataset

df = pd.read_csv(DATA_PATH, sep = ',') # Read the csv file into a DataFrame

assert "stroke" in df.columns, "Expected 'stroke' column." # Check if the DataFrame has a column named 'sroke'

print(df.shape)
df.head() # Display first 5 rows of the DataFrame to give a quick preview of the data

(5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


# Initial Exploration

In [20]:

display(df.describe(include='all')) # Give summary statistics for the dataset and ensures it includes both numeric and categorical columns

print("\nMissing values per column:") 

print(df.isna().sum().sort_values(ascending=False)) # See which columns might need cleaning or imputation.

# Dataset size: 5110 rows.
# Missing values: Only the bmi column has missing entries — 4909 non-null values out of 5110, meaning 201 BMI values are missing.
# Numeric columns: age, hypertension, heart_disease, avg_glucose_level, bmi, and stroke are numeric.
# Categorical columns: gender, ever_married, work_type, Residence_type, and smoking_status.
# Target column: stroke (0 = no stroke, 1 = stroke).
# Imbalance: Only about 5% had strokes (mean ≈ 0.0487) — this means the dataset is highly imbalanced.

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
count,5110.000000,5110,5110.000000,5110.000000,5110.000000,5110,5110,5110,5110.000000,4909.000000,5110,5110.000000
unique,NaN,3,NaN,NaN,NaN,2,5,2,NaN,NaN,4,NaN
top,NaN,Female,NaN,NaN,NaN,Yes,Private,Urban,NaN,NaN,never smoked,NaN
freq,NaN,2994,NaN,NaN,NaN,3353,2925,2596,NaN,NaN,1892,NaN
mean,36517.829354,NaN,43.226614,0.097456,0.054012,NaN,NaN,NaN,106.147677,28.893237,NaN,0.048728
std,21161.721625,NaN,22.612647,0.296607,0.226063,NaN,NaN,NaN,45.283560,7.854067,NaN,0.215320
min,67.000000,NaN,0.080000,0.000000,0.000000,NaN,NaN,NaN,55.120000,10.300000,NaN,0.000000
25%,17741.250000,NaN,25.000000,0.000000,0.000000,NaN,NaN,NaN,77.245000,23.500000,NaN,0.000000
50%,36932.000000,NaN,45.000000,0.000000,0.000000,NaN,NaN,NaN,91.885000,28.100000,NaN,0.000000
75%,54682.000000,NaN,61.000000,0.000000,0.000000,NaN,NaN,NaN,114.090000,33.100000,NaN,0.000000



Missing values per column:
bmi                  201
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
smoking_status         0
stroke                 0
dtype: int64


# Preprocessing

In [ ]:
# Drop unnecessary columns (id)
X = df.drop(columns =["id"]) 

# Encode categorical variables (gender, ever_married, work_type, Residence_type and smoking_status)
enc = OrdinalEncoder()
enc.fit(X)

# Handle missing values (bmi) using 'Nearest neighbours imputation'
imp_median = SimpleImputer(missing_values = np.nan, strategy = 'median')
imp_median.fit(X)

# Encode categorical variables (gender, work_type, etc.)

# Scale numeric features

ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'Male'

# Classification: predict stroke vs no-stroke

In [15]:
# Binary label from quality with an adjustable threshold
#threshold = 7
X = df.drop(columns=["stroke"])
y_clf = df['stroke'] 

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.2, stratify=y_clf, random_state=RNG
)

clf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, C=1.0, class_weight=None, random_state=RNG))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
cv_clf = cross_validate(
    clf_pipe, X_train_c, y_train_c,
    cv=skf, scoring=("accuracy", "f1", "roc_auc"),
    return_train_score=True, n_jobs=-1
)

print(f"CV Mean Acc (train): {cv_clf['train_accuracy'].mean():.3f} ± {cv_clf['train_accuracy'].std():.3f}")
print(f"CV Mean Acc (val):   {cv_clf['test_accuracy'].mean():.3f}  ± {cv_clf['test_accuracy'].std():.3f}")
print(f"CV Mean F1 (val):    {cv_clf['test_f1'].mean():.3f}")
print(f"CV Mean ROC-AUC (val): {cv_clf['test_roc_auc'].mean():.3f}")

# Fit & evaluate on TEST
t0 = time.time()
clf_pipe.fit(X_train_c, y_train_c)
train_time = time.time() - t0

t1 = time.time()
y_proba = clf_pipe.predict_proba(X_test_c)[:, 1]
y_pred_c = (y_proba >= 0.5).astype(int)
pred_time = time.time() - t1

acc = accuracy_score(y_test_c, y_pred_c)
f1 = f1_score(y_test_c, y_pred_c, zero_division=0)
roc = roc_auc_score(y_test_c, y_proba)
print(f"Test Accuracy: {acc:.3f} | Test F1: {f1:.3f} | Test ROC-AUC: {roc:.3f}")
print(f"Train time: {train_time*1000:.1f} ms | Predict time: {pred_time*1000:.1f} ms")

# Confusion matrix
cm = confusion_matrix(y_test_c, y_pred_c)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
plt.figure()
disp.plot(values_format='d')
plt.title(f"Confusion Matrix")
plt.show()


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\patri\anaconda3\envs\ml_course\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\patri\anaconda3\envs\ml_course\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\patri\anaconda3\envs\ml_course\Lib\site-packages\sklearn\pipeline.py", line 655, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "c:\Users\patri\anaconda3\envs\ml_course\Lib\site-packages\sklearn\pipeline.py", line 5

# Define Features and Target

# Train-Test Split

# Model Pipeline

# Cross-Validation

# Fit Model on Training Data

# Evaluate on Test Data

# Confusion Matrix

# Reproducibility and Reporting



In [14]:
import sklearn, sys
print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

print("\nPipelines used:")
print("Classification:", clf_pipe)


Python: 3.13.7 | packaged by Anaconda, Inc. | (main, Sep  9 2025, 19:54:37) [MSC v.1929 64 bit (AMD64)]
NumPy: 2.3.3
Pandas: 2.3.2
scikit-learn: 1.7.2

Pipelines used:
Classification: Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=2000, random_state=42))])


### Mini Model Card

- **Objective:**  Predict whether a patient will experience a stroke based on health and demographic features.

- **Data:** Stroke dataset with 5110 entries; features include age, gender, hypertension, heart disease, marital status, work type, residence type, average glucose level, BMI, and smoking status; CSV is comma-separated.

- **Validation:** 5-fold stratified cross-validation on the training set; stratified train-test split to preserve class balance; final evaluation on an untouched test set.

- **Key metrics:** Reported metrics include accuracy, F1 score, and ROC-AUC. Example numbers 

- **Strengths / limitations:** Logistic regression provides interpretability; however, class imbalance (~5% stroke cases) may reduce sensitivity for minority class. Preprocessing required for missing BMI values and categorical encoding.

- **Ethical notes / misuse risks:** Predictions depend on dataset demographics and may not generalize to other populations; model is not a medical diagnostic tool and should not be used for clinical decision-making without expert oversight.